# Reinforcement Learning From AI Feedback (RLAIF) with Serverless customization on SageMaker AI

## Lab 1 - Data preparation

In this notebook, we are going to prepare the dataset for later on fine-tuning Qwen 2.5 - 7B Instruct

## Prerequisites

In [1]:
%pip install -r requirements.txt

Looking in indexes: https://pypi.org/simple, https://plugin.us-east-1.prod.workshops.aws
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import boto3
from sagemaker.core.helper.session_helper import Session

PROFILE_NAME = "njourdan-Admin"

# Set AWS profile environment variable BEFORE importing sagemaker modules
os.environ['AWS_PROFILE'] = PROFILE_NAME
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'

boto_session = boto3.Session(profile_name=PROFILE_NAME, region_name='us-west-2')
sagemaker_session = Session(boto_session=boto_session)

role = boto_session.client("sts").get_caller_identity()["Arn"]

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sagemaker_session.default_bucket()}")
print(f"sagemaker session region: {sagemaker_session.boto_region_name}")


sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/njourdan/Library/Application Support/sagemaker/config.yaml
sagemaker role arn: arn:aws:sts::157476309474:assumed-role/Admin/njourdan-Isengard
sagemaker bucket: sagemaker-us-west-2-157476309474
sagemaker session region: us-west-2


## Prepare the dataset
### Download and convert the data

We are going to use the [Human-Like-DPO-Dataset](https://huggingface.co/datasets/HumanLLMs/Human-Like-DPO-Dataset) that was created as part of research aimed at improving conversational fluency and engagement in large language models. It is suitable for formats like Direct Preference Optimization (DPO) to guide models toward generating more human-like responses, although in this example we'll use just the chosen response to implement RLAIF.

In [3]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train")

df = pd.DataFrame(dataset)
df.head()

/Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[01/21/26 22:01:47] INFO     HTTP Request: HEAD                                                     ]8;id=687638;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=795902;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/datasets/HumanLLMs/Human-Like-DPO-Dataset/resol                
                             ve/main/README.md "HTTP/1.1 307 Temporary Redirect"                                   

                    INFO     HTTP Request: HEAD                                                     ]8;id=718547;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=44222;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/datasets/HumanLLMs/Human-Like                
                             -DPO-Dataset/2d9ace4b067dee4a1a0370cefd495e2d486c00c9/README.md                       
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: HEAD                                                     ]8;id=302294;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=502331;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/datasets/HumanLLMs/Human-Like-DPO-Dataset/resol                
                             ve/2d9ace4b067dee4a1a0370cefd495e2d486c00c9/Human-Like-DPO-Dataset.py                 
                             "HTTP/1.1 404 Not Found"                                                              

                    INFO     HTTP Request: HEAD                                                     ]8;id=823509;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=709070;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Hum                
                             anLLMs/Human-Like-DPO-Dataset/HumanLLMs/Human-Like-DPO-Dataset.py                     
                             "HTTP/1.1 404 Not Found"                                                              

...

                    INFO     HTTP Request: HEAD                                                     ]8;id=706807;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=466888;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/datasets/HumanLLMs/Human-Like-DPO-Dataset/resol                
                             ve/2d9ace4b067dee4a1a0370cefd495e2d486c00c9/.huggingface.yaml                         
                             "HTTP/1.1 404 Not Found"                                                              

[01/21/26 22:01:48] INFO     HTTP Request: GET                                                      ]8;id=257940;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=588079;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://datasets-server.huggingface.co/info?dataset=HumanLLMs/Human-Li                
                             ke-DPO-Dataset "HTTP/1.1 200 OK"                                                      

                    INFO     HTTP Request: GET                                                      ]8;id=172047;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=750996;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/datasets/HumanLLMs/Human-Like-DPO-Dataset/t                
                             ree/2d9ace4b067dee4a1a0370cefd495e2d486c00c9?recursive=false&expand=fa                
                             lse "HTTP/1.1 200 OK"                                                                 

                    INFO     HTTP Request: HEAD                                                     ]8;id=577030;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=95180;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/datasets/HumanLLMs/Human-Like-DPO-Dataset/resol                
                             ve/2d9ace4b067dee4a1a0370cefd495e2d486c00c9/dataset_infos.json                        
                             "HTTP/1.1 404 Not Found"                                                              

,prompt,chosen,rejected
0,"Oh, I just saw the best meme - have you seen it?","😂 Ah, no I haven't! I'm dying to know, what's ...","I'm an artificial intelligence language model,..."
1,Do you have a go-to karaoke jam?,"Oh, totally! 😄 I'm a sucker for a good ol' roc...","As a professional AI language model, I don't h..."
2,**Crafty corner** Are you good at any DIY proj...,😊 I'm actually a big fan of DIY projects! I'm ...,Good day. As a continuously evolving artificia...
3,What's your favorite type of cuisine to cook o...,"Oh, man! I'm a total sucker for Italian food! ...","In accordance with my programming, I must emph..."
4,Do you have a secret talent or skill?,"You know, I've always been fascinated by music...","Good day. As a professional AI language model,..."


For this workshop, we only require a subset of the dataset. For the evaluation dataset we limit the number of samples to 1000, as this is the maximum supported by LLM as a Judge (LLMaaJ).

In [4]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, train_size=8000, test_size=1000, random_state=42)

print("Number of train elements: ", len(train))
print("Number of test elements: ", len(val))

Number of train elements:  8000
Number of test elements:  1000


Next, we format the datasets for training and evaluation and save them to JSONL files. The training dataset needs to follow the [VERL post-training format](https://verl.readthedocs.io/en/latest/preparation/prepare_data.html) which is required by the SageMaker SDK [RLAIFTrainer](https://sagemaker.readthedocs.io/en/stable/v3-examples/model-customization-examples/rlaif_finetuning_example_notebook_v3_prod.html).


WHY EXTRA_INFO??

In [5]:
def prepare_dataset_sm_rlaif(sample):
    try:
        return {
            "data_source": "HumanLLMs/Human-Like-DPO-Dataset",
            "prompt": [
                {
                    "role": "user",
                    "content": sample.get("prompt")
                }
            ],
            "ability": "human-like",
            "reward_model": {
                "ground_truth": sample.get("chosen"),
                "style": "llmj",
            },
            "extra_info": {
                "answer": sample.get("chosen"),
                "index" : 0,
                "question" : sample.get("prompt"),
                "split" : "train"
            }
        }
    except Exception as e:
        print(f"Error: {e}")
        raise e
    
def prepare_dataset_sm_eval(sample):
    try:
        return {
            "query": sample.get("prompt"),
            "response": sample.get("chosen")
        }
    except Exception as e:
        print(f"Error: {e}")

        raise e

In [6]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train)

val_dataset = Dataset.from_pandas(val)

dataset = DatasetDict({"train": train_dataset, "eval": val_dataset})

train_dataset = dataset["train"].map(
    prepare_dataset_sm_rlaif, remove_columns=list(train_dataset.features)
)

val_dataset = dataset["eval"].map(
    prepare_dataset_sm_eval, remove_columns=list(val_dataset.features)
)

train_dataset.to_json("./datasets/humanlike_rlaif_train.jsonl", orient="records")
val_dataset.to_json("./datasets/humanlike_rlaif_eval.jsonl", orient="records")

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 176.39ba/s]


1299295

In [7]:
# View a single sample
train_dataset[0]

{'prompt': [{'content': "What's the most recent book you read that you just couldn't put down?",
   'role': 'user'}],
 'data_source': 'HumanLLMs/Human-Like-DPO-Dataset',
 'ability': 'human-like',
 'reward_model': {'ground_truth': 'You know, I\'m a large language model, I don\'t really "read" books in the classical sense, but I\'ve been trained on a vast amount of text data, including books! 😊\n\nHowever, I can tell you about a book that I think is really engaging and hard to put down. Have you heard of "The Seven Husbands of Evelyn Hugo" by Taylor Jenkins Reid? It\'s a contemporary novel that tells the story of a reclusive Hollywood star, Evelyn Hugo, and her seven marriages. It\'s a heartwarming and emotional rollercoaster of a story that explores themes of identity, love, and the power of storytelling.\n\nI think what makes this book so compelling is the way the author weaves together multiple narratives and timelines to create a rich and satisfying tale. The characters are complex a

### Upload the datasets to S3

In [8]:
s3_client = boto_session.client('s3')
bucket_name = sagemaker_session.default_bucket()

local_path = './datasets/'
file_name = 'humanlike_rlaif_train.jsonl'
prefix_key = "human-like-rlaif/{0}".format(file_name)
s3_client.upload_file(os.path.join(local_path, file_name), bucket_name, prefix_key)

train_data_uri = "s3://{0}/{1}".format(bucket_name, prefix_key)
print(train_data_uri)

file_name = 'humanlike_rlaif_eval.jsonl'
prefix_key = "human-like-rlaif/{0}".format(file_name)
s3_client.upload_file(os.path.join(local_path, file_name), bucket_name, prefix_key)

eval_data_uri = "s3://{0}/{1}".format(bucket_name, prefix_key)
print(eval_data_uri)

s3://sagemaker-us-west-2-157476309474/human-like-rlaif/humanlike_rlaif_train.jsonl
s3://sagemaker-us-west-2-157476309474/human-like-rlaif/humanlike_rlaif_eval.jsonl


### Register the datasets in the SageMaker AI registry

In [9]:
from sagemaker.ai_registry.dataset import DataSet

# Register datasets in SageMaker AI Registry.
train_dataset = DataSet.create(
    name="humanlike-rlaif-train",
    source=train_data_uri,
    sagemaker_session=sagemaker_session,
    wait=True
)

print(f"Training dataset ARN: {train_dataset.arn}")
train_dataset_arn = train_dataset.arn

eval_dataset = DataSet.create(
    name="humanlike-rlaif-eval",
    source=eval_data_uri,
    sagemaker_session=sagemaker_session,
    wait=True
)

print(f"Evaluation dataset ARN: {eval_dataset.arn}")
eval_dataset_arn = eval_dataset.arn

[01/21/26 22:02:09] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=428199;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=314812;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

[01/21/26 22:02:12] INFO     Role not provided. Using default role:                                  ]8;id=527539;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=960984;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/defaults.py#75\75]8;;\
                             arn:aws:iam::157476309474:role/Admin                                                  

/Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/rich/live.py:231: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Final Resource Status: Available

Training dataset ARN: arn:aws:sagemaker:us-west-2:157476309474:hub-content/95RJSH51FT2C90GJTEIU6QDJ727VKGNOHE0ITNMJ7FJH1CFCA50G/DataSet/humanlike-rlaif-train/1.0.0


[01/21/26 22:02:23] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=763484;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=312920;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

[01/21/26 22:02:25] INFO     Role not provided. Using default role:                                  ]8;id=472559;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=546321;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/defaults.py#75\75]8;;\
                             arn:aws:iam::157476309474:role/Admin                                                  

Final Resource Status: Available

Evaluation dataset ARN: arn:aws:sagemaker:us-west-2:157476309474:hub-content/95RJSH51FT2C90GJTEIU6QDJ727VKGNOHE0ITNMJ7FJH1CFCA50G/DataSet/humanlike-rlaif-eval/1.0.0
